# Proyecto final IA (IS-484): KNN para la navegación de un robot

**Integrantes:** COMPLETAR nombre 1 (usuario de GitHub) · COMPLETAR nombre 2 (usuario de GitHub)

Este notebook sigue el orden del enunciado de la docente:

| Sección | Qué contiene |
| --- | --- |
| 1. Problema y datos | Problema, dataset de UCI y verificación de los requisitos |
| 2. Algoritmo: KNN | Cómo funciona, distancia y estandarización |
| 3. Cálculo a mano | 9 filas + 1 consulta, paso a paso, y verificación con código |
| 4. Modelo y evaluación | División 80/20, validación cruzada para K y métricas |
| 5. Segundo algoritmo | Árbol de decisión con `max_depth` ajustado |
| 6. Comparación y conclusión | Mismas métricas para los dos modelos y decisión final |

> Las celdas marcadas con **COMPLETAR** son trabajo del equipo (cálculo a mano, observaciones y conclusión).
> El notebook debe entregarse **completamente ejecutado** (Kernel → Restart & Run All) y subido con sus salidas.

In [ ]:
import os
import time
import zipfile
import urllib.request
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

pd.set_option("display.width", 120)
pd.set_option("display.max_columns", None)

SEMILLA = 42
COLUMNAS = ["SD_front", "SD_left", "SD_right", "SD_back"]
OBJETIVO = "Class"

## 1. Problema y datos

**Problema.** Un robot móvil debe decidir qué movimiento hacer en cada instante para seguir una pared sin chocar.
Se plantea como **clasificación multiclase**: dadas 4 distancias medidas por sus sensores, predecir la acción.

- **Entrada:** `SD_front`, `SD_left`, `SD_right`, `SD_back` (lectura mínima en un arco de 60° al frente, a la izquierda, a la derecha y detrás del robot).
- **Salida:** `Class`, una de 4 acciones: `Move-Forward`, `Slight-Right-Turn`, `Sharp-Right-Turn`, `Slight-Left-Turn`.

**Dataset.** [UCI Wall-Following Robot Navigation Data](https://archive.ics.uci.edu/dataset/194/wall+following+robot+navigation+data),
archivo `sensor_readings_4.data` (versión de 4 atributos).

In [ ]:
# El notebook está en notebooks/ y los datos en data/
RUTA = next((p for p in ["../data/sensor_readings_4.data", "data/sensor_readings_4.data"] if os.path.exists(p)),
            "../data/sensor_readings_4.data")
URL_UCI = "https://archive.ics.uci.edu/static/public/194/wall+following+robot+navigation+data.zip"

if not os.path.exists(RUTA):
    print("No se encontró el archivo; intentando descargarlo desde UCI...")
    try:
        carpeta = os.path.dirname(RUTA)
        os.makedirs(carpeta, exist_ok=True)
        ruta_zip = os.path.join(carpeta, "uci_wall_following.zip")
        urllib.request.urlretrieve(URL_UCI, ruta_zip)
        with zipfile.ZipFile(ruta_zip) as z:
            nombre = [n for n in z.namelist() if n.endswith("sensor_readings_4.data")][0]
            with z.open(nombre) as origen, open(RUTA, "wb") as destino:
                destino.write(origen.read())
        os.remove(ruta_zip)
    except Exception as error:
        raise FileNotFoundError(
            "Descarga sensor_readings_4.data desde https://archive.ics.uci.edu/dataset/194 "
            "y guárdalo en la carpeta data/ del repositorio."
        ) from error

df = pd.read_csv(RUTA, header=None, names=COLUMNAS + [OBJETIVO])
df[COLUMNAS] = df[COLUMNAS].apply(pd.to_numeric, errors="coerce")   # por si el archivo trajera encabezado
df = df.dropna().reset_index(drop=True)
df[OBJETIVO] = df[OBJETIVO].astype(str).str.strip()
print("Archivo leído:", RUTA)
print("Filas y columnas:", df.shape)
df.head()

In [ ]:
print("Valores faltantes por columna:")
print(df.isna().sum().to_string())
print("\nFilas duplicadas:", int(df.duplicated().sum()))
print("\nResumen de las variables:")
display(df[COLUMNAS].describe().round(3))

In [ ]:
conteo = df[OBJETIVO].value_counts()
porcentaje = (conteo / len(df) * 100).round(2)
display(pd.DataFrame({"casos": conteo, "porcentaje (%)": porcentaje}))

# Requisitos de la docente
assert len(df) >= 500, "El dataset debe tener al menos 500 filas"
assert conteo.min() >= 50, "La clase menos frecuente debe tener al menos 50 casos"
print(f"Cumple: {len(df)} filas (mínimo 500) y la clase menor tiene {conteo.min()} casos (mínimo 50).")

conteo.plot(kind="bar", figsize=(7, 3.5), color="tab:blue")
plt.ylabel("Casos")
plt.title("Casos por clase")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

**Observaciones de la exploración (COMPLETAR).**
Escribe aquí 2 o 3 observaciones con números, por ejemplo: qué clase domina, qué tan pequeña es la menor (el desbalance) y qué implica para elegir métricas.

## 2. Algoritmo: KNN

KNN no aprende una ecuación: **guarda los ejemplos de entrenamiento**. Para clasificar una lectura nueva sigue este recorrido:

lectura nueva → distancias → K vecinos más cercanos → votación → clase

La cercanía se mide con la **distancia euclidiana** entre las 4 lecturas de dos casos $x$ y $z$:

$$d(x, z) = \sqrt{(x_1 - z_1)^2 + (x_2 - z_2)^2 + (x_3 - z_3)^2 + (x_4 - z_4)^2}$$

Antes de medir distancias se **estandarizan** las variables, con la media $\mu$ y la desviación $\sigma$ calculadas solo con el entrenamiento:

$$z = \frac{x - \mu}{\sigma}$$

- **K pequeño** (por ejemplo 1): decide con muy pocos casos y memoriza el ruido (sobreajuste).
- **K muy grande**: casi todo el dataset vota y se ignoran los detalles (subajuste).
- Cada vecino aporta **un voto** a su clase (aquí hay 4 clases posibles).

In [ ]:
# ¿Por qué estandarizar? Mira si las 4 variables tienen rangos comparables.
rangos = df[COLUMNAS].agg(["min", "max", "mean", "std"]).T
rangos["rango"] = rangos["max"] - rangos["min"]
display(rangos.round(3))

**¿Por qué estandarizar? (COMPLETAR).** Con los rangos de arriba, explica si alguna variable dominaría la distancia sin estandarizar.

## 3. Cálculo a mano

Se usan **10 filas del dataset: 9 de entrenamiento y 1 lectura nueva (la consulta)**. El formato es el de la guía 8:
en cada paso se escribe la **fórmula**, se **reemplazan los valores** y se anota el **resultado**, con un "¿Por qué?".

Las filas se eligen con semilla fija (reproducible), cuidando que: las 9 filas tengan al menos 3 de las 4 clases,
no haya empates al votar con K = 1, 3 y 5, y la predicción cambie entre K = 1 y K = 5.

In [ ]:
def elegir_muestra(datos, semilla_inicial=SEMILLA, intentos=500):
    for exigir_cambio in (True, False):
        for s in range(semilla_inicial, semilla_inicial + intentos):
            muestra = datos.sample(10, random_state=s)
            ent, nueva = muestra.iloc[:9], muestra.iloc[9][COLUMNAS].astype(float)
            if ent[OBJETIVO].nunique() < 3:
                continue
            mu, sigma = ent[COLUMNAS].mean(), ent[COLUMNAS].std(ddof=0)
            if (sigma == 0).any():
                continue
            z_ent = (ent[COLUMNAS] - mu) / sigma
            z_nueva = (nueva - mu) / sigma
            dist = np.sqrt(((z_ent - z_nueva) ** 2).sum(axis=1)).sort_values()
            sin_empates, predicciones = True, {}
            for k in (1, 3, 5):
                votos = ent.loc[dist.index[:k], OBJETIVO].value_counts()
                if len(votos) > 1 and votos.iloc[0] == votos.iloc[1]:
                    sin_empates = False
                    break
                predicciones[k] = votos.index[0]
            if not sin_empates:
                continue
            if exigir_cambio and predicciones[1] == predicciones[5]:
                continue
            return s, ent, muestra.iloc[[9]]
    raise RuntimeError("No se encontró una muestra adecuada; cambia la semilla inicial.")


semilla_muestra, entrenamiento, consulta = elegir_muestra(df)
print("Semilla usada:", semilla_muestra)
print("Posiciones de las 9 filas en el dataset (anótalas en el README):", entrenamiento.index.tolist())
print("Posición de la consulta:", consulta.index.tolist())

tabla_9 = entrenamiento.copy()
tabla_9.index = [f"x{i}" for i in range(1, 10)]
print("\nPaso 1. Las 9 filas de entrenamiento:")
display(tabla_9)
print("Lectura nueva (consulta):")
display(consulta.rename(index={consulta.index[0]: "q"}))

### Paso 2. Media y desviación de cada variable

Con las 9 filas de entrenamiento, para cada variable $j$ (la desviación se divide entre $n = 9$, igual que `StandardScaler`):

$$\mu_j = \frac{1}{9}\sum_{i=1}^{9} x_{ij} \qquad\qquad \sigma_j = \sqrt{\frac{1}{9}\sum_{i=1}^{9}\left(x_{ij} - \mu_j\right)^2}$$

**¿Por qué? (COMPLETAR).**

In [ ]:
# PASO 2 (a mano). Escribe cada fórmula con los valores reemplazados de la tabla; Python solo hace la aritmética.
# Ejemplo para SD_front (reemplaza x1..x9 por los 9 valores de la tabla):
#   mu_front = (x1 + x2 + x3 + x4 + x5 + x6 + x7 + x8 + x9) / 9
#   sigma_front = (((x1 - mu_front)**2 + (x2 - mu_front)**2 + ... + (x9 - mu_front)**2) / 9) ** 0.5

mu_mano = None      # COMPLETAR: [mu_front, mu_left, mu_right, mu_back]
sigma_mano = None   # COMPLETAR: [sigma_front, sigma_left, sigma_right, sigma_back]

### Paso 3. Estandarización

$$z_{ij} = \frac{x_{ij} - \mu_j}{\sigma_j}$$

Se aplica a las 9 filas y también a la consulta (con la misma $\mu_j$ y $\sigma_j$ del entrenamiento).

**¿Por qué? (COMPLETAR).**

In [ ]:
# PASO 3 (a mano). Ejemplo para la fila x1 y SD_front:
#   z_x1_front = (x1_front - mu_front) / sigma_front

Z_mano = None            # COMPLETAR: lista de 9 filas, cada una con 4 valores z
z_consulta_mano = None   # COMPLETAR: lista con los 4 valores z de la consulta

### Paso 4. Distancia euclidiana de la consulta a cada fila

$$d(q, x_i) = \sqrt{(z_{q1} - z_{i1})^2 + (z_{q2} - z_{i2})^2 + (z_{q3} - z_{i3})^2 + (z_{q4} - z_{i4})^2}$$

**¿Por qué? (COMPLETAR).**

In [ ]:
# PASO 4 (a mano). Ejemplo para la fila x1:
#   d_x1 = ((zq1 - z11)**2 + (zq2 - z12)**2 + (zq3 - z13)**2 + (zq4 - z14)**2) ** 0.5

d_mano = None   # COMPLETAR: lista con las 9 distancias, en el orden x1..x9

### Paso 5. Vecinos y votación

Se ordenan las 9 distancias de menor a mayor. Con **K = 1, 3 y 5** se toman los K vecinos más cercanos y gana la clase con más votos.

**¿Por qué cambia (o no) la predicción al cambiar K? (COMPLETAR).**

In [ ]:
# PASO 5 (a mano). Escribe la predicción para cada K como texto, por ejemplo "Move-Forward".
predicciones_mano = {1: None, 3: None, 5: None}   # COMPLETAR

### Paso 6. Verificación con código

Ahora se repite todo con NumPy y se compara con scikit-learn (`kneighbors`). Las distancias deben coincidir con las del cálculo a mano.

In [ ]:
X_9 = entrenamiento[COLUMNAS].to_numpy()
y_9 = entrenamiento[OBJETIVO].to_numpy()
q = consulta[COLUMNAS].to_numpy()[0]

mu = X_9.mean(axis=0)
sigma = X_9.std(axis=0)            # ddof = 0 (dividir entre n), igual que StandardScaler
Z = (X_9 - mu) / sigma
zq = (q - mu) / sigma
d = np.sqrt(((Z - zq) ** 2).sum(axis=1))
orden = np.argsort(d)

print("Media:", mu.round(4))
print("Desviación:", sigma.round(4))
tabla_verif = pd.DataFrame(Z.round(4), columns=[f"z_{c}" for c in COLUMNAS], index=[f"x{i}" for i in range(1, 10)])
tabla_verif["clase"] = y_9
tabla_verif["distancia"] = d.round(4)
print("\nDistancias a la consulta (ordenadas):")
display(tabla_verif.sort_values("distancia"))
print("z de la consulta:", zq.round(4))

predicciones = {}
for k in (1, 3, 5):
    votos = Counter(y_9[orden[:k]])
    predicciones[k] = votos.most_common(1)[0][0]
    print(f"K = {k}: votos {dict(votos)} -> predicción: {predicciones[k]}")

# Comprobación con scikit-learn
escalador_9 = StandardScaler().fit(entrenamiento[COLUMNAS])
knn_9 = KNeighborsClassifier(n_neighbors=5).fit(escalador_9.transform(entrenamiento[COLUMNAS]), y_9)
dist_sk, _ = knn_9.kneighbors(escalador_9.transform(consulta[COLUMNAS]))
print("\nDistancias NumPy vs scikit-learn (5 menores):", np.allclose(np.sort(d)[:5], dist_sk[0]))
print("Predicción de scikit-learn (K = 5):", knn_9.predict(escalador_9.transform(consulta[COLUMNAS]))[0])

In [ ]:
def comparar(nombre, mano, codigo, tol=1e-3):
    if mano is None:
        print(f"{nombre}: pendiente de completar a mano")
    else:
        try:
            coincide = np.allclose(np.array(mano, dtype=float), codigo, atol=tol)
        except (ValueError, TypeError):
            print(f"{nombre}: el formato no es el esperado, revisa la forma de tu lista")
            return
        print(f"{nombre}: {'coincide con el código' if coincide else 'NO coincide, revisa el cálculo a mano'}")


comparar("Paso 2, medias", mu_mano, mu)
comparar("Paso 2, desviaciones", sigma_mano, sigma)
comparar("Paso 3, z de las 9 filas", Z_mano, Z)
comparar("Paso 3, z de la consulta", z_consulta_mano, zq)
comparar("Paso 4, distancias", d_mano, d)
for k in (1, 3, 5):
    if predicciones_mano[k] is None:
        print(f"Paso 5, K = {k}: pendiente de completar a mano")
    else:
        print(f"Paso 5, K = {k}: {'coincide' if predicciones_mano[k] == predicciones[k] else 'NO coincide, revisa el voto'}")

**Para la sustentación (COMPLETAR).** Si la docente pide K = 7 en vivo con estas 9 filas, ¿qué pasa con los vecinos y con la predicción? (Pista: K = 7 ya vota con casi todo el conjunto.)

## 4. Modelo y evaluación

Se usa el dataset completo, dividido en **80 % entrenamiento y 20 % prueba** de forma estratificada (las clases están desbalanceadas).
K se elige con **validación cruzada de 5 particiones solo sobre el entrenamiento**; la prueba se usa una única vez al final.

Como las clases están desbalanceadas, el criterio para elegir K es el **F1 macro** (promedio del F1 de cada clase, sin favorecer a la más frecuente), y se reporta junto a la accuracy.

In [ ]:
X = df[COLUMNAS]
y = df[OBJETIVO]
X_ent, X_prueba, y_ent, y_prueba = train_test_split(X, y, test_size=0.2, random_state=SEMILLA, stratify=y)
print("Entrenamiento:", X_ent.shape, "| Prueba:", X_prueba.shape)
display(pd.DataFrame({"entrenamiento": y_ent.value_counts(normalize=True).round(4),
                      "prueba": y_prueba.value_counts(normalize=True).round(4)}))

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
clases = sorted(y.unique())

In [ ]:
# Efecto de la estandarización (K = 5), medido con validación cruzada sobre el entrenamiento
modelos_escala = {
    "Sin estandarizar": KNeighborsClassifier(n_neighbors=5),
    "Con StandardScaler": make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
}
filas = []
for nombre, modelo in modelos_escala.items():
    puntajes = cross_val_score(modelo, X_ent, y_ent, cv=cv, scoring="f1_macro")
    filas.append({"modelo (K = 5)": nombre, "F1 macro (CV, media)": round(puntajes.mean(), 4), "desviación": round(puntajes.std(), 4)})
display(pd.DataFrame(filas))

In [ ]:
# Validación cruzada para elegir K (impares de 1 a 51)
malla_k = {"kneighborsclassifier__n_neighbors": list(range(1, 52, 2))}
busqueda_knn = GridSearchCV(make_pipeline(StandardScaler(), KNeighborsClassifier()), malla_k,
                            cv=cv, scoring="f1_macro", return_train_score=True, n_jobs=-1)
busqueda_knn.fit(X_ent, y_ent)
k_mejor = busqueda_knn.best_params_["kneighborsclassifier__n_neighbors"]
print(f"K elegido: {k_mejor} | F1 macro de validación: {busqueda_knn.best_score_:.4f}")

res = pd.DataFrame(busqueda_knn.cv_results_)
ks = res["param_kneighborsclassifier__n_neighbors"].astype(int)
plt.figure(figsize=(8, 4.2))
plt.plot(ks, res["mean_train_score"], marker=".", label="Entrenamiento")
plt.plot(ks, res["mean_test_score"], marker=".", label="Validación cruzada")
plt.axvline(k_mejor, color="gray", linestyle="--", label=f"K elegido = {k_mejor}")
plt.xlabel("K (número de vecinos)")
plt.ylabel("F1 macro")
plt.title("F1 macro según K")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Interpretación de la curva (COMPLETAR).** Describe qué pasa con K = 1 (entrenamiento vs validación), qué pasa con K grande y por qué el K elegido está en el medio.

In [ ]:
# Evaluación final con la prueba (una sola vez)
mejor_knn = busqueda_knn.best_estimator_
t0 = time.perf_counter()
pred_knn = mejor_knn.predict(X_prueba)
t_knn = time.perf_counter() - t0

print(f"KNN (K = {k_mejor})")
print(f"Accuracy en prueba: {accuracy_score(y_prueba, pred_knn):.4f}")
print(f"F1 macro en prueba: {f1_score(y_prueba, pred_knn, average='macro'):.4f}\n")
print(classification_report(y_prueba, pred_knn, labels=clases, zero_division=0, digits=3))

fig, ax = plt.subplots(figsize=(5.5, 5))
ConfusionMatrixDisplay.from_predictions(y_prueba, pred_knn, labels=clases, xticks_rotation=30, cmap="Blues", ax=ax)
ax.set_title(f"KNN (K = {k_mejor}): matriz de confusión")
plt.tight_layout()
plt.show()

**Interpretación de las métricas (COMPLETAR).** ¿Qué clases se confunden más? ¿Cómo le va a la clase menos frecuente? ¿Por qué el F1 macro y la accuracy pueden diferir?

## 5. Segundo algoritmo: árbol de decisión

Un árbol clasifica haciendo preguntas del tipo "¿variable ≤ corte?" y elige las que dejan los grupos más puros (guía 8).
Se evalúa con **la misma división, la misma validación cruzada y las mismas métricas**. El parámetro a ajustar es `max_depth` (1 a 15).
El árbol no necesita estandarizar: decide por cortes y no por distancias.

In [ ]:
malla_arbol = {"max_depth": list(range(1, 16))}
busqueda_arbol = GridSearchCV(DecisionTreeClassifier(random_state=SEMILLA), malla_arbol,
                              cv=cv, scoring="f1_macro", return_train_score=True, n_jobs=-1)
busqueda_arbol.fit(X_ent, y_ent)
profundidad_mejor = busqueda_arbol.best_params_["max_depth"]
print(f"max_depth elegido: {profundidad_mejor} | F1 macro de validación: {busqueda_arbol.best_score_:.4f}")

res_a = pd.DataFrame(busqueda_arbol.cv_results_)
plt.figure(figsize=(8, 4.2))
plt.plot(res_a["param_max_depth"].astype(int), res_a["mean_train_score"], marker="o", label="Entrenamiento")
plt.plot(res_a["param_max_depth"].astype(int), res_a["mean_test_score"], marker="o", label="Validación cruzada")
plt.axvline(profundidad_mejor, color="gray", linestyle="--", label=f"max_depth elegido = {profundidad_mejor}")
plt.xlabel("max_depth (profundidad máxima)")
plt.ylabel("F1 macro")
plt.title("F1 macro según la profundidad del árbol")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
mejor_arbol = busqueda_arbol.best_estimator_
t0 = time.perf_counter()
pred_arbol = mejor_arbol.predict(X_prueba)
t_arbol = time.perf_counter() - t0

print(f"Árbol de decisión (max_depth = {profundidad_mejor})")
print(f"Accuracy en prueba: {accuracy_score(y_prueba, pred_arbol):.4f}")
print(f"F1 macro en prueba: {f1_score(y_prueba, pred_arbol, average='macro'):.4f}\n")
print(classification_report(y_prueba, pred_arbol, labels=clases, zero_division=0, digits=3))

print("Importancia de cada variable:")
display(pd.Series(mejor_arbol.feature_importances_, index=COLUMNAS).sort_values(ascending=False).round(4).to_frame("importancia"))
print("Primeras reglas del árbol (hasta profundidad 3):")
print(export_text(mejor_arbol, feature_names=COLUMNAS, max_depth=3))

## 6. Comparación y conclusión

Los dos modelos se comparan con la misma división de datos, la misma validación cruzada y las mismas métricas.

In [ ]:
def fila(nombre, parametro, busqueda, pred, tiempo):
    i = busqueda.best_index_
    return {
        "modelo": nombre,
        "parámetro elegido": parametro,
        "F1 macro (CV)": round(busqueda.best_score_, 4),
        "desv. CV": round(busqueda.cv_results_["std_test_score"][i], 4),
        "accuracy (prueba)": round(accuracy_score(y_prueba, pred), 4),
        "F1 macro (prueba)": round(f1_score(y_prueba, pred, average="macro"), 4),
        "tiempo de predicción (s)": round(tiempo, 4),
    }


tabla_comparacion = pd.DataFrame([
    fila("KNN", f"K = {k_mejor}", busqueda_knn, pred_knn, t_knn),
    fila("Árbol de decisión", f"max_depth = {profundidad_mejor}", busqueda_arbol, pred_arbol, t_arbol),
])
display(tabla_comparacion)

f1_clases = pd.DataFrame({
    "KNN": f1_score(y_prueba, pred_knn, labels=clases, average=None),
    "Árbol": f1_score(y_prueba, pred_arbol, labels=clases, average=None),
}, index=clases).round(4)
print("F1 por clase en prueba:")
display(f1_clases)

In [ ]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4.8))
ConfusionMatrixDisplay.from_predictions(y_prueba, pred_knn, labels=clases, xticks_rotation=30, cmap="Blues", ax=ejes[0], colorbar=False)
ejes[0].set_title(f"KNN (K = {k_mejor})")
ConfusionMatrixDisplay.from_predictions(y_prueba, pred_arbol, labels=clases, xticks_rotation=30, cmap="Greens", ax=ejes[1], colorbar=False)
ejes[1].set_title(f"Árbol (max_depth = {profundidad_mejor})")
plt.tight_layout()
plt.show()

mejor_modelo = tabla_comparacion.loc[tabla_comparacion["F1 macro (prueba)"].idxmax(), "modelo"]
clase_menor = df[OBJETIVO].value_counts().idxmin()
print(f"Mayor F1 macro en prueba: {mejor_modelo}")
print(f"F1 en la clase menos frecuente ({clase_menor}):")
print(f1_clases.loc[clase_menor].to_string())

### Conclusión (COMPLETAR)

Responde con los números de las tablas de arriba, no de memoria:

1. **¿Qué modelo elegirían y por qué?** Compara el F1 macro en prueba y la estabilidad en la validación cruzada (desviación).
2. **¿Cuál falla menos en la clase menos frecuente?** Usa el F1 por clase y las matrices de confusión.
3. **Costo de decidir.** KNN guarda todos los datos y calcula distancias en cada predicción; el árbol responde con unas pocas preguntas y se lee como reglas. ¿Qué pesa más en un robot que decide en tiempo real?
4. **Limitaciones.** Una sola división de prueba, un solo dataset, un solo robot y un solo recorrido.

### Preguntas para la sustentación

- ¿Por qué se estandariza y por qué la media y la desviación se calculan solo con el entrenamiento?
- ¿Qué hace KNN con K = 1 y con un K enorme? ¿Qué muestra la curva de la sección 4?
- ¿Por qué se usa F1 macro y no solo accuracy en este dataset?
- Si la docente cambia K de 5 a 15 (o `max_depth` de 3 a 10), ¿qué esperas que pase **antes** de ejecutar?
- ¿Cuál es la diferencia entre la validación cruzada y el conjunto de prueba?